In [8]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA = PROJECT_ROOT / "data" / "raw" / "YRBS_2007.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"
SUMMARY_DIR = PROJECT_ROOT / "outputs" / "summary"
REFERENCE_DIR = PROJECT_ROOT / "references"

for folder in [PROCESSED_DIR, TABLE_DIR, FIGURE_DIR, SUMMARY_DIR, REFERENCE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RAW_DATA

WindowsPath('C:/Users/User/project-cycle-3(1)/project-cycle-3/data/raw/YRBS_2007.csv')

In [9]:
raw = pd.read_csv(RAW_DATA)

print("Raw dataset shape:", raw.shape)
raw[["CurrentCigaretteUse", "BMIPCT"]].head()

Raw dataset shape: (14041, 103)


,CurrentCigaretteUse,BMIPCT
0,7.0,NaN
1,NaN,66.531824
2,NaN,NaN
3,1.0,98.174319
4,1.0,NaN


In [10]:
required_columns = ["CurrentCigaretteUse", "BMIPCT"]
missing_required = [col for col in required_columns if col not in raw.columns]

if missing_required:
    raise ValueError(f"Missing required column(s): {missing_required}")

missing_summary = raw[required_columns].isna().sum().to_frame("missing_count")
current_cigarette_codes = raw["CurrentCigaretteUse"].value_counts(dropna=False).sort_index()

print("Missing values:")
display(missing_summary)

print("Original CurrentCigaretteUse code counts:")
display(current_cigarette_codes)

Missing values:


,missing_count
CurrentCigaretteUse,718
BMIPCT,979


Original CurrentCigaretteUse code counts:


CurrentCigaretteUse
1.0    10734
2.0      753
3.0      375
4.0      250
5.0      295
6.0      229
7.0      687
NaN      718
Name: count, dtype: int64

In [11]:
selected = raw[["CurrentCigaretteUse", "BMIPCT"]].copy()

selected = selected.rename(columns={
    "CurrentCigaretteUse": "current_cigarette_use_code",
    "BMIPCT": "bmi_percentile"
})

selected["current_cigarette_use"] = np.select(
    [
        selected["current_cigarette_use_code"].eq(1),
        selected["current_cigarette_use_code"].between(2, 7)
    ],
    [
        0,
        1
    ],
    default=np.nan
)

selected["cigarette_group"] = selected["current_cigarette_use"].map({
    0: "Non-current cigarette user",
    1: "Current cigarette user"
})

selected["bmi_percentile"] = pd.to_numeric(selected["bmi_percentile"], errors="coerce")

valid_bmi = selected["bmi_percentile"].between(0, 100)
complete_analysis_row = (
    selected["current_cigarette_use"].notna()
    & selected["bmi_percentile"].notna()
    & valid_bmi
)

clean = selected.loc[complete_analysis_row].copy()
clean["current_cigarette_use"] = clean["current_cigarette_use"].astype(int)

clean = clean[[
    "current_cigarette_use_code",
    "current_cigarette_use",
    "cigarette_group",
    "bmi_percentile"
]]

clean.head()

,current_cigarette_use_code,current_cigarette_use,cigarette_group,bmi_percentile
3,1.0,0,Non-current cigarette user,98.174319
5,1.0,0,Non-current cigarette user,33.075531
6,1.0,0,Non-current cigarette user,45.688334
7,1.0,0,Non-current cigarette user,62.390331
8,1.0,0,Non-current cigarette user,13.969642


In [12]:
#This summary shows how many rows were excluded and how many students remain in each group. 
cleaning_summary = pd.DataFrame({
    "item": [
        "Rows in raw dataset",
        "Rows with missing CurrentCigaretteUse",
        "Rows with invalid CurrentCigaretteUse code outside 1-7",
        "Rows with missing BMIPCT",
        "Rows with BMIPCT outside 0-100",
        "Rows kept in clean analysis dataset",
        "Rows removed from analysis dataset"
    ],
    "count": [
        len(raw),
        selected["current_cigarette_use_code"].isna().sum(),
        selected["current_cigarette_use_code"].notna().sum()
        - selected["current_cigarette_use_code"].isin([1, 2, 3, 4, 5, 6, 7]).sum(),
        selected["bmi_percentile"].isna().sum(),
        (~valid_bmi & selected["bmi_percentile"].notna()).sum(),
        len(clean),
        len(raw) - len(clean)
    ]
})

group_counts = clean["cigarette_group"].value_counts().rename_axis("cigarette_group").reset_index(name="n")

display(cleaning_summary)
display(group_counts)

,item,count
0,Rows in raw dataset,14041
1,Rows with missing CurrentCigaretteUse,718
2,Rows with invalid CurrentCigaretteUse code out...,0
3,Rows with missing BMIPCT,979
4,Rows with BMIPCT outside 0-100,0
5,Rows kept in clean analysis dataset,12437
6,Rows removed from analysis dataset,1604


,cigarette_group,n
0,Non-current cigarette user,10003
1,Current cigarette user,2434


In [13]:
processed_path = PROCESSED_DIR / "yrbs_2007_cigarette_bmi_clean.csv"
clean.to_csv(processed_path, index=False)

cleaning_summary_path = TABLE_DIR / "cleaning_summary.csv"
group_counts_path = TABLE_DIR / "clean_group_counts.csv"

cleaning_summary.to_csv(cleaning_summary_path, index=False)
group_counts.to_csv(group_counts_path, index=False)

recoding_notes = '''# Recoding Notes

Research question: Is the mean BMI percentile different between students who currently use cigarettes and those who do not?

Group variable: CurrentCigaretteUse

Recoding:
- Original code 1 -> 0 = Non-current cigarette user
- Original codes 2 through 7 -> 1 = Current cigarette user
- Missing or invalid CurrentCigaretteUse values are excluded

Response variable:
- BMIPCT = BMI percentile
- Missing BMIPCT values are excluded
- BMI percentile values are kept only if they are between 0 and 100

Method planned:
- Welch two-sample t-test
- 95% confidence interval for the difference in mean BMI percentile
- Difference order: current cigarette users minus non-current cigarette users
'''

notes_path = REFERENCE_DIR / "variable_notes.md"
notes_path.write_text(recoding_notes, encoding="utf-8")

print("Saved clean data to:", processed_path)
print("Saved cleaning summary to:", cleaning_summary_path)
print("Saved group counts to:", group_counts_path)
print("Saved recoding notes to:", notes_path)

Saved clean data to: C:\Users\User\project-cycle-3(1)\project-cycle-3\data\processed\yrbs_2007_cigarette_bmi_clean.csv
Saved cleaning summary to: C:\Users\User\project-cycle-3(1)\project-cycle-3\outputs\tables\cleaning_summary.csv
Saved group counts to: C:\Users\User\project-cycle-3(1)\project-cycle-3\outputs\tables\clean_group_counts.csv
Saved recoding notes to: C:\Users\User\project-cycle-3(1)\project-cycle-3\references\variable_notes.md
